# E-commerce Data Analysis and Sales Prediction Using Machine Learning
**Internship:** AICTE | IBM SkillsBuild Data Analytics with AI | BharatCares 2026

---

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Load Dataset
Dataset from Kaggle: https://www.kaggle.com/datasets/thedevastator/unlock-profits-with-e-commerce-sales-data

In [ ]:
# NOTE: Download dataset from Kaggle and place 'Amazon Sale Report.csv' in same folder
# OR we create sample data below if file not found

try:
    df = pd.read_csv('Amazon Sale Report.csv', encoding='unicode_escape')
    print('Dataset loaded from file!')
except FileNotFoundError:
    print('File not found - Creating sample dataset for demo...')
    np.random.seed(42)
    n = 500
    categories = ['T-shirt', 'Shirt', 'Blazer', 'Trousers', 'Kurta']
    sizes = ['S', 'M', 'L', 'XL', 'XXL']
    states = ['Maharashtra', 'Karnataka', 'Tamil Nadu', 'Delhi', 'Gujarat']
    df = pd.DataFrame({
        'Order ID': ['ORD' + str(i) for i in range(1, n+1)],
        'Category': np.random.choice(categories, n),
        'Size': np.random.choice(sizes, n),
        'Qty': np.random.randint(1, 10, n),
        'Amount': np.random.randint(200, 5000, n),
        'ship-state': np.random.choice(states, n),
        'Status': np.random.choice(['Shipped', 'Delivered', 'Cancelled'], n),
        'Date': pd.date_range(start='2022-01-01', periods=n, freq='D').strftime('%m-%d-%Y')
    })
    print('Sample dataset created with', n, 'records!')

print('Shape:', df.shape)
df.head()

## Step 3: Data Exploration (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Basic Statistics ===')
print(df.describe())

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

## Step 4: Data Cleaning

In [ ]:
# Drop rows with missing Amount or Qty
df = df.dropna(subset=['Amount', 'Qty'])

# Fill other missing values
df.fillna('Unknown', inplace=True)

# Convert Amount and Qty to numeric
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce').fillna(1)

# Remove cancelled orders for prediction
df_clean = df[df['Status'] != 'Cancelled'].copy()

print('Cleaned dataset shape:', df_clean.shape)
print('Data cleaned successfully!')

## Step 5: Data Visualization

In [ ]:
# Plot 1: Sales by Category
plt.figure(figsize=(10, 5))
category_sales = df_clean.groupby('Category')['Amount'].sum().sort_values(ascending=False)
sns.barplot(x=category_sales.index, y=category_sales.values, palette='Blues_d')
plt.title('Total Sales by Category')
plt.xlabel('Category')
plt.ylabel('Total Amount (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('sales_by_category.png')
plt.show()
print('Chart saved!')

In [ ]:
# Plot 2: Sales by Size
plt.figure(figsize=(8, 5))
size_sales = df_clean.groupby('Size')['Qty'].sum().sort_values(ascending=False)
sns.barplot(x=size_sales.index, y=size_sales.values, palette='Greens_d')
plt.title('Units Sold by Size')
plt.xlabel('Size')
plt.ylabel('Total Quantity')
plt.tight_layout()
plt.savefig('sales_by_size.png')
plt.show()

In [ ]:
# Plot 3: Top States by Revenue
plt.figure(figsize=(10, 5))
state_sales = df_clean.groupby('ship-state')['Amount'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=state_sales.index, y=state_sales.values, palette='Oranges_d')
plt.title('Top 10 States by Revenue')
plt.xlabel('State')
plt.ylabel('Total Revenue (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('sales_by_state.png')
plt.show()

In [ ]:
# Plot 4: Amount Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['Amount'], bins=30, kde=True, color='steelblue')
plt.title('Sales Amount Distribution')
plt.xlabel('Amount (INR)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('amount_distribution.png')
plt.show()

## Step 6: Feature Engineering

In [ ]:
# Encode categorical columns
le = LabelEncoder()
df_model = df_clean.copy()

for col in ['Category', 'Size', 'ship-state', 'Status']:
    if col in df_model.columns:
        df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))

# Select features
feature_cols = [c for c in df_model.columns if '_encoded' in c] + ['Qty']
X = df_model[feature_cols]
y = df_model['Amount']

print('Features used:', feature_cols)
print('X shape:', X.shape)
print('y shape:', y.shape)

## Step 7: Model Training — Sales Prediction

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

In [ ]:
# Model 1: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print('=== Linear Regression Results ===')
print('MAE :', round(mean_absolute_error(y_test, lr_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2))
print('R2  :', round(r2_score(y_test, lr_pred), 4))

In [ ]:
# Model 2: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('=== Random Forest Results ===')
print('MAE :', round(mean_absolute_error(y_test, rf_pred), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2))
print('R2  :', round(r2_score(y_test, rf_pred), 4))

## Step 8: Model Comparison

In [ ]:
# Compare Models
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [
        round(mean_absolute_error(y_test, lr_pred), 2),
        round(mean_absolute_error(y_test, rf_pred), 2)
    ],
    'RMSE': [
        round(np.sqrt(mean_squared_error(y_test, lr_pred)), 2),
        round(np.sqrt(mean_squared_error(y_test, rf_pred)), 2)
    ],
    'R2 Score': [
        round(r2_score(y_test, lr_pred), 4),
        round(r2_score(y_test, rf_pred), 4)
    ]
})

print('=== Model Comparison ===')
print(results.to_string(index=False))

In [ ]:
# Actual vs Predicted Plot (Random Forest)
plt.figure(figsize=(8, 5))
plt.scatter(y_test, rf_pred, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Amount')
plt.ylabel('Predicted Amount')
plt.title('Actual vs Predicted Sales (Random Forest)')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png')
plt.show()

## Step 9: Feature Importance

In [ ]:
# Feature Importance
importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index, palette='viridis')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

print('Most important feature:', importance.idxmax())

## Step 10: Conclusion

In [ ]:
print('=== PROJECT SUMMARY ===')
print('Project: E-commerce Data Analysis and Sales Prediction Using Machine Learning')
print('Dataset : Amazon E-commerce Sales Data (Kaggle)')
print('Models  : Linear Regression, Random Forest Regressor')
print('Best    : Random Forest (higher R2 score)')
print('')
print('Key Findings:')
print('1. Top selling categories identified')
print('2. State-wise revenue analysis completed')
print('3. Sales prediction model built with ML')
print('4. Random Forest outperforms Linear Regression')
print('')
print('Project completed successfully!')